# 01 — Prepare data and cache the teacher (run ONCE, about 1.5–3 h)

**Notebook settings (right panel):**
1. *Accelerator*: **GPU T4 x2** (one GPU is used here).
2. *Internet*: **On** (needs a phone-verified Kaggle account).
3. *Add Input* → *Datasets*: add one copy of **ISIC 2019** (the training images, ~25k `ISIC_*.jpg`)
   and one copy of **PAD-UFES-20** (has `metadata.csv` with a `diagnostic` column and `PAT_*.png` images).

Then click **Save Version → Save & Run All (Commit)**. You can close the browser; it runs in the background.
When it finishes, its *Output* is the input for notebook 02.

In [ ]:
!nvidia-smi -L
%cd /kaggle/working
!rm -rf IEEE-TMI && git clone -q https://github.com/Nafi14078/IEEE-TMI.git
%cd /kaggle/working/IEEE-TMI
!pip install -q open_clip_torch
!pip install -q -e . --no-deps

In [ ]:
# Locate the datasets you added as inputs.
import os, urllib.request
import pandas as pd

def count_images(top, prefix):
    return sum(f.startswith(prefix) for _, _, fs in os.walk(top) for f in fs)

tops = [os.path.join('/kaggle/input', d) for d in os.listdir('/kaggle/input')]
isic_root = max(tops, key=lambda t: count_images(t, 'ISIC_'))
print('ISIC images found under', isic_root, count_images(isic_root, 'ISIC_'))
assert count_images(isic_root, 'ISIC_') > 20000, 'Add an ISIC 2019 dataset as input'

pad_meta = None
for t in tops:
    for root, _, fs in os.walk(t):
        if 'metadata.csv' in fs:
            cols = pd.read_csv(os.path.join(root, 'metadata.csv'), nrows=1).columns
            if 'diagnostic' in cols and 'img_id' in cols:
                pad_meta, pad_root = os.path.join(root, 'metadata.csv'), t
assert pad_meta, 'Add a PAD-UFES-20 dataset as input'
print('PAD-UFES-20:', pad_meta, count_images(pad_root, 'PAT_'), 'images')

# Official ISIC 2019 label + metadata CSVs (small), so lesion_id grouping is always available.
os.makedirs('/kaggle/tmp_raw', exist_ok=True)
base = 'https://isic-challenge-data.s3.amazonaws.com/2019/'
for f in ['ISIC_2019_Training_GroundTruth.csv', 'ISIC_2019_Training_Metadata.csv']:
    dst = f'/kaggle/tmp_raw/{f}'
    if not os.path.exists(dst):
        urllib.request.urlretrieve(base + f, dst)
print('ISIC CSVs downloaded')

In [ ]:
# Task 3: freeze splits (lesion-grouped 70/10/20, nested 1/5/10 % labels x 3 seeds)
!python scripts/prepare_splits.py --isic-gt /kaggle/tmp_raw/ISIC_2019_Training_GroundTruth.csv --isic-meta /kaggle/tmp_raw/ISIC_2019_Training_Metadata.csv --isic-images "{isic_root}" --pad-meta "{pad_meta}" --pad-images "{pad_root}"

In [ ]:
# Resize once (short side 256 px)
!python scripts/preprocess_images.py --workers 4
!du -sh data/processed

In [ ]:
# Task 9: cache BiomedCLIP outputs (8 views per image + 8 prompt sets)
!python scripts/cache_teacher.py --teacher hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224 --out cache/biomedclip.pt --views 8 --batch 64 --workers 4

In [ ]:
# Task 4: does the label-free reliability predict when the teacher is right?  (GO / NO-GO)
for f in (1, 5, 10):
    !python scripts/check_reliability.py --set data.label_fraction={f} data.num_workers=4

In [ ]:
# Teacher baselines (zero-shot + linear probe) for every label fraction and seed
!python scripts/run_grid.py teacher
!rm -rf /kaggle/tmp_raw
!du -sh data cache runs results